# 네이버 뉴스 API 데이터 수집

**사전 준비**
1. 프로젝트 루트에 `.env` 파일을 생성하고 API 키를 입력한다. (`.env.example` 참고)
2. [네이버 개발자센터](https://developers.naver.com/apps) 에서 애플리케이션을 등록하고 **검색 API** 를 활성화한다.


`NAVER_CLIENT_ID`=발급받은_Client_ID  
`NAVER_CLIENT_SECRET`=발급받은_Client_Secret

수집 결과는 `data/naver_news_collected.csv` 로 저장된다.  
`tokenizer_comparison.ipynb` 에서 학습 데이터로 사용된다.

In [ ]:
# ============================================================
# 0. 필요 라이브러리 설치
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("python-dotenv")
pip_install("requests")
pip_install("pandas")
pip_install("matplotlib")
pip_install("seaborn")

print("라이브러리 설치 완료")

In [ ]:
# ============================================================
# 1. 공통 임포트 및 환경 변수 로드
# ============================================================
import os, time, html, re, warnings
from datetime import datetime
from collections import Counter

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# 한글 고딕 폰트를 설정한다.
for font_name in ["AppleGothic", "NanumGothic", "Nanum Gothic", "Malgun Gothic"]:
    if any(font_name in f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = font_name
        print(f"폰트 설정: {font_name}")
        break
plt.rcParams["axes.unicode_minus"] = False

# 출력 디렉토리를 생성한다.
os.makedirs("data",    exist_ok=True)
os.makedirs("figures", exist_ok=True)

# .env 파일에서 API 키를 로드한다.
load_dotenv()
CLIENT_ID     = os.environ.get("NAVER_CLIENT_ID", "")
CLIENT_SECRET = os.environ.get("NAVER_CLIENT_SECRET", "")

if not CLIENT_ID or not CLIENT_SECRET:
    print("⚠️  .env 파일에 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 이 설정되지 않았다.")
else:
    print(f"API 키 로드 완료 (Client ID: {CLIENT_ID[:4]}*****)")

NEWS_CSV = "data/naver_news_collected.csv"

In [ ]:
# ============================================================
# 2. 수집 설정
#    KEYWORDS : 수집할 키워드 목록 (경제 + 다양한 도메인)
#               도메인 다양성 확보 → 10% vs 100% 훈련 시 어휘 커버리지 차이 발생
#               → UNK Rate 변화를 의미 있게 관찰할 수 있다
#    DISPLAY  : 키워드 1개당 최대 수집 건수 (최대 100)
#    PAGES    : 페이지 수 (DISPLAY × PAGES 건 수집)
# ============================================================
KEYWORDS = [
    # 주식 / 증시
    "코스피", "코스닥", "주식", "삼성전자 주가", "개인투자자",
    "공매도", "배당주", "ETF", "PER", "시가총액",
    # 부동산
    "부동산", "아파트 매매", "전세", "월세", "청약",
    "재개발", "재건축", "분양가", "갭투자", "역전세",
    # 가상자산 / 코인
    "비트코인", "이더리움", "코인", "가상화폐", "업비트",
    "알트코인", "스테이블코인", "NFT", "코인 규제", "암호화폐",
    # 거시경제
    "금리", "환율", "인플레이션", "경기침체", "무역수지",
    "한국은행", "기준금리", "미국 연준", "달러", "국채",
    # IT / 테크
    "인공지능", "LLM", "ChatGPT", "반도체", "클라우드",
    "사이버보안", "자율주행", "로봇", "양자컴퓨터", "스타트업",
    # 정치 / 사회
    "국회", "대통령", "총선", "정책", "복지",
    "저출산", "고령화", "이민", "교육개혁", "의료파업",
    # 문화 / 엔터
    "K팝", "드라마", "영화", "웹툰", "유튜브",
    "OTT", "게임", "아이돌", "넷플릭스", "콘텐츠",
    # 스포츠
    "야구", "축구", "농구", "올림픽", "손흥민",
    "KBO", "EPL", "월드컵", "마라톤", "e스포츠",
    # 과학 / 환경
    "기후변화", "탄소중립", "신재생에너지", "우주개발", "바이오",
    "백신", "신약", "줄기세포", "핵융합", "해양오염",
]

DISPLAY  = 100   # 키워드당 1회 요청 건수 (최대 100)
PAGES    = 3     # 페이지 수 → 키워드당 최대 DISPLAY × PAGES 건
                 # 90키워드 × 300건 = 최대 27,000건 → 중복 제거 후 약 1.5MB 이상 확보
DELAY    = 0.5   # 요청 간 딜레이 (초) — API 과부하 방지

API_URL  = "https://openapi.naver.com/v1/search/news.json"

print(f"수집 키워드 {len(KEYWORDS)}개 / 키워드당 최대 {DISPLAY * PAGES}건")
print(f"예상 총 수집량: 최대 {len(KEYWORDS) * DISPLAY * PAGES:,}건")
print(f"예상 소요 시간: 약 {len(KEYWORDS) * PAGES * DELAY / 60:.0f}분")

In [ ]:
# ============================================================
# 3. 뉴스 수집 함수
#    fetch_news()   : 단일 키워드 단일 페이지 요청
#    collect_news() : 전체 키워드 × 전체 페이지 반복 수집
# ============================================================
def clean_text(s: str) -> str:
    """HTML 태그·엔티티를 제거하고 공백을 정리한다."""
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def fetch_news(keyword: str, display: int = 100, start: int = 1) -> list:
    """
    네이버 뉴스 검색 API 단일 요청.
    반환: [{"keyword", "title", "description", "pubDate", "link"}, ...]
    """
    headers = {
        "X-Naver-Client-Id":     CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
    }
    params = {
        "query":   keyword,
        "display": display,
        "start":   start,
        "sort":    "date",
    }
    try:
        resp = requests.get(API_URL, headers=headers, params=params, timeout=10)
        resp.raise_for_status()
        items = resp.json().get("items", [])
        records = []
        for item in items:
            records.append({
                "keyword":     keyword,
                "title":       clean_text(item.get("title", "")),
                "description": clean_text(item.get("description", "")),
                "pubDate":     item.get("pubDate", ""),
                "link":        item.get("originallink") or item.get("link", ""),
            })
        return records
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP 오류 [{keyword}]: {e}")
        return []
    except Exception as e:
        print(f"  요청 실패 [{keyword}]: {e}")
        return []


def collect_news(keywords: list, display: int, pages: int, delay: float) -> list:
    """전체 키워드 × 페이지를 수집하고 중복을 제거한 레코드 리스트를 반환한다."""
    all_records = []
    seen_titles = set()
    total_req = len(keywords) * pages
    req_count = 0

    for kw in keywords:
        for page in range(pages):
            start     = page * display + 1
            req_count += 1
            print(f"  [{req_count:>3}/{total_req}] '{kw}'  start={start} ...", end=" ")

            records = fetch_news(kw, display=display, start=start)
            new_records = [r for r in records if r["title"] not in seen_titles]
            for r in new_records:
                seen_titles.add(r["title"])

            all_records.extend(new_records)
            print(f"+{len(new_records)}건  (누계 {len(all_records):,}건)")
            time.sleep(delay)

    return all_records


print("수집 함수 정의 완료")

In [ ]:
# ============================================================
# 4. 데이터 수집 실행 및 CSV 저장
# ============================================================
if not CLIENT_ID or not CLIENT_SECRET:
    print("⚠️  API 키가 설정되지 않아 수집을 건너뛴다.")
    print("    .env 파일에 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 을 설정 후 재실행한다.")
    news_records = []
else:
    print(f"수집 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    news_records = collect_news(KEYWORDS, display=DISPLAY, pages=PAGES, delay=DELAY)
    print(f"\n수집 완료: 총 {len(news_records):,}건 (중복 제거 후)")

    df_news = pd.DataFrame(news_records)
    print("\n[샘플 5건]")
    display(df_news.head())

    df_news.to_csv(NEWS_CSV, index=False, encoding="utf-8-sig")
    print(f"\nCSV 저장 완료: {NEWS_CSV}")
    print(f"  - 전체 행 수  : {len(df_news):,}")
    print(f"  - 파일 크기   : {df_news.to_csv(index=False).encode('utf-8').__len__()/1024:.1f} KB")

In [ ]:
# ============================================================
# 5. 수집 데이터 탐색적 분석 (EDA)
# ============================================================
if not news_records:
    print("수집 데이터가 없어 EDA 를 건너뛴다.")
else:
    df_eda = pd.DataFrame(news_records)

    # 날짜 파싱
    df_eda["date"] = pd.to_datetime(
        df_eda["pubDate"], format="%a, %d %b %Y %H:%M:%S +0900", errors="coerce"
    ).dt.date

    # 도메인 매핑 (키워드 → 도메인)
    DOMAIN_MAP = {
        **{kw: "주식/증시"   for kw in ["코스피","코스닥","주식","삼성전자 주가","개인투자자","공매도","배당주","ETF","PER","시가총액"]},
        **{kw: "부동산"      for kw in ["부동산","아파트 매매","전세","월세","청약","재개발","재건축","분양가","갭투자","역전세"]},
        **{kw: "가상자산"    for kw in ["비트코인","이더리움","코인","가상화폐","업비트","알트코인","스테이블코인","NFT","코인 규제","암호화폐"]},
        **{kw: "거시경제"    for kw in ["금리","환율","인플레이션","경기침체","무역수지","한국은행","기준금리","미국 연준","달러","국채"]},
        **{kw: "IT/테크"     for kw in ["인공지능","LLM","ChatGPT","반도체","클라우드","사이버보안","자율주행","로봇","양자컴퓨터","스타트업"]},
        **{kw: "정치/사회"   for kw in ["국회","대통령","총선","정책","복지","저출산","고령화","이민","교육개혁","의료파업"]},
        **{kw: "문화/엔터"   for kw in ["K팝","드라마","영화","웹툰","유튜브","OTT","게임","아이돌","넷플릭스","콘텐츠"]},
        **{kw: "스포츠"      for kw in ["야구","축구","농구","올림픽","손흥민","KBO","EPL","월드컵","마라톤","e스포츠"]},
        **{kw: "과학/환경"   for kw in ["기후변화","탄소중립","신재생에너지","우주개발","바이오","백신","신약","줄기세포","핵융합","해양오염"]},
    }
    df_eda["domain"] = df_eda["keyword"].map(DOMAIN_MAP).fillna("기타")
    df_eda["desc_len"] = df_eda["description"].str.len()

    # 기본 통계 출력
    print(f"총 수집 건수       : {len(df_eda):,}")
    print(f"수집 키워드 수     : {df_eda['keyword'].nunique()}")
    print(f"description 평균   : {df_eda['desc_len'].mean():.1f}자")
    print(f"description 중앙값 : {df_eda['desc_len'].median():.1f}자")
    print(f"최소 / 최대 길이   : {df_eda['desc_len'].min()} / {df_eda['desc_len'].max()}자")

    # ── (1) 키워드별 수집 건수 Top 20 ──────────────────────────
    kw_counts = df_eda["keyword"].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(kw_counts.index[::-1], kw_counts.values[::-1], color="#4C72B0")
    for i, v in enumerate(kw_counts.values[::-1]):
        ax.text(v + 0.5, i, str(v), va="center", fontsize=8)
    ax.set_title("키워드별 수집 건수 Top 20")
    ax.set_xlabel("건수")
    plt.tight_layout()
    plt.savefig("figures/fig_eda_keyword.png", dpi=150)
    plt.show()

    # ── (2) 도메인별 수집 건수 ─────────────────────────────────
    domain_counts = df_eda["domain"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = plt.cm.Set2.colors[:len(domain_counts)]
    ax.bar(domain_counts.index, domain_counts.values, color=colors)
    for i, v in enumerate(domain_counts.values):
        ax.text(i, v + 10, str(v), ha="center", fontsize=8)
    ax.set_title("도메인별 수집 건수")
    ax.set_ylabel("건수")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig("figures/fig_eda_domain.png", dpi=150)
    plt.show()

    # ── (3) 날짜별 기사 수 (최근 14일) ────────────────────────
    date_counts = df_eda["date"].value_counts().sort_index().tail(14)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(date_counts.index.astype(str), date_counts.values, color="#DD8452", edgecolor="white")
    ax.set_title("날짜별 기사 수 (최근 14일)")
    ax.set_xlabel("날짜")
    ax.set_ylabel("건수")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig("figures/fig_eda_date.png", dpi=150)
    plt.show()

    # ── (4) description 길이 분포 ─────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df_eda["desc_len"], bins=50, color="#55A868", edgecolor="white")
    ax.axvline(df_eda["desc_len"].mean(),   color="red",    linestyle="--", label=f"평균 {df_eda['desc_len'].mean():.0f}자")
    ax.axvline(df_eda["desc_len"].median(), color="orange", linestyle="--", label=f"중앙값 {df_eda['desc_len'].median():.0f}자")
    ax.set_title("기사 description 길이 분포")
    ax.set_xlabel("문자 수")
    ax.set_ylabel("빈도")
    ax.legend()
    plt.tight_layout()
    plt.savefig("figures/fig_eda_desc_len.png", dpi=150)
    plt.show()

    # ── (5) 도메인별 description 길이 박스플롯 ────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    order = df_eda.groupby("domain")["desc_len"].median().sort_values(ascending=False).index
    sns.boxplot(data=df_eda, x="domain", y="desc_len", order=order,
                palette="Set2", ax=ax)
    ax.set_title("도메인별 기사 길이 분포")
    ax.set_xlabel("도메인")
    ax.set_ylabel("문자 수")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig("figures/fig_eda_domain_len.png", dpi=150)
    plt.show()

    # ── (6) 상위 빈출 단어 Top 50 (불용어 제거) ───────────────
    stopwords = {"의","을","를","이","가","은","는","과","와","에","에서","로","으로","도","만","하","한","및","등","이번","지난","오는", "17일", "18일", "19일", "2026", "3월", "있다", "있는"}
    word_counts = Counter(
        w for text in df_eda["description"]
        for w in re.sub(r"[^가-힣a-zA-Z0-9]", " ", text).split()
        if w not in stopwords and len(w) > 1
    )
    top_words = word_counts.most_common(50)
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.barh([w for w, _ in top_words[::-1]], [c for _, c in top_words[::-1]], color="coral")
    ax.set_title("전체 기사 빈출 단어 Top 50")
    ax.set_xlabel("빈도")
    plt.tight_layout()
    plt.savefig("figures/fig_eda_top_words.png", dpi=150)
    plt.show()

    print("\nEDA 그래프 저장 완료: figures/fig_eda_*.png")